In [ ]:
import pandas as pd

df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print("\nTipos de datos:")
print(df.dtypes)

In [ ]:
# Diagnóstico de nulos
print("=== Nulos por columna ===")
print(df.isnull().sum())

print("\n=== TotalCharges — valores únicos problemáticos ===")
# Convertimos forzado y vemos qué explota
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Nulos en TotalCharges después de conversión: {df['TotalCharges'].isnull().sum()}")

print("\n=== Distribución de Churn (variable objetivo) ===")
print(df['Churn'].value_counts())
print(f"\nTasa de churn: {df['Churn'].value_counts(normalize=True)['Yes']:.1%}")

In [ ]:
# Ver las filas con TotalCharges nulo
print("=== Clientes con TotalCharges nulo ===")
print(df[df['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']])

print("\n=== Estadísticas columnas numéricas ===")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe())

In [ ]:
# Imputamos con criterio de negocio — tenure 0 = sin facturación
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verificamos
print(f"Nulos restantes en TotalCharges: {df['TotalCharges'].isnull().sum()}")

print("\n=== Estadísticas columnas numéricas ===")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe())

In [ ]:
# Churn por método de pago
print("=== Tasa de churn por método de pago ===")
print(df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).unstack())

print("\n=== Tasa de churn por tipo de contrato ===")
print(df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack())

In [ ]:
import matplotlib.pyplot as plt

# Tenure vs Churn — el comportamiento en el tiempo
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gráfico 1 — Distribución de tenure por Churn
df.groupby('Churn')['tenure'].plot(
    kind='hist', 
    bins=30, 
    alpha=0.6, 
    ax=axes[0],
    legend=True
)
axes[0].set_title('Tenure por Churn')
axes[0].set_xlabel('Meses como cliente')

# Gráfico 2 — MonthlyCharges vs Churn
df.groupby('Churn')['MonthlyCharges'].plot(
    kind='hist',
    bins=30,
    alpha=0.6,
    ax=axes[1],
    legend=True
)
axes[1].set_title('Cargo mensual por Churn')
axes[1].set_xlabel('USD/mes')

# Gráfico 3 — Tenure promedio por contrato
df.groupby('Contract')['tenure'].mean().plot(
    kind='bar',
    ax=axes[2],
    color=['#ff6b6b', '#ffc947', '#00d4aa']
)
axes[2].set_title('Tenure promedio por contrato')
axes[2].set_xlabel('')
axes[2].set_ylabel('Meses promedio')

plt.tight_layout()
plt.show()

# Números detrás de los gráficos
print("=== Tenure promedio por grupo ===")
print(df.groupby('Churn')['tenure'].mean().round(1))

print("\n=== MonthlyCharges promedio por grupo ===")
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

In [ ]:
# ===========================================
# HALLAZGOS EDA — Telco Customer Churn
# Fecha: 2026-06-12
# ===========================================
# 1. Dataset: 7,043 clientes, 21 columnas, churn 26.5%
# 2. TotalCharges: 11 nulos por tenure=0, imputados con 0
# 3. Predictores clave identificados:
#    - Contract: month-to-month = 42.7% churn vs 2.8% two-year
#    - PaymentMethod: electronic check = 45.3% churn
#    - Tenure: clientes que se van promedian 18 meses vs 37.6
#    - MonthlyCharges: clientes que se van pagan $74 vs $61
# 4. Segmento dormido: ~$20/mes, sin internet, churn casi cero
# 5. Paradoja premium: Fiber optic = mayor costo = mayor churn
# ===========================================

In [ ]:
import numpy as np

# Array de cargos mensuales — simulando 5 clientes
charges = np.array([20.5, 70.3, 45.8, 99.1, 55.0])

print(f"Tipo: {type(charges)}")
print(f"Dtype: {charges.dtype}")
print(f"Shape: {charges.shape}")
print(f"Datos: {charges}")

In [ ]:
import time

# Datos grandes — 1 millón de cargos simulados
big_charges = np.random.uniform(20, 120, 1_000_000)

# Método 1 — loop Python
start = time.time()
result_loop = []
for c in big_charges:
    result_loop.append(c * 1.10)
loop_time = time.time() - start

# Método 2 — vectorización NumPy
start = time.time()
result_numpy = big_charges * 1.10
numpy_time = time.time() - start

print(f"Loop Python:      {loop_time:.4f} segundos")
print(f"NumPy vectorizado: {numpy_time:.4f} segundos")
print(f"NumPy es {loop_time/numpy_time:.0f}x más rápido")

In [ ]:
# Usamos los MonthlyCharges reales del dataset
charges = df['MonthlyCharges'].values  # .values convierte columna Pandas a array NumPy

print(f"Tipo: {type(charges)}")
print(f"Shape: {charges.shape}")
print(f"\n--- Estadísticas vectorizadas ---")
print(f"Promedio:    ${np.mean(charges):.2f}")
print(f"Mediana:     ${np.median(charges):.2f}")
print(f"Desv. std:   ${np.std(charges):.2f}")
print(f"Mínimo:      ${np.min(charges):.2f}")
print(f"Máximo:      ${np.max(charges):.2f}")

print(f"\n--- Operaciones de negocio ---")
# ¿Cuántos clientes pagan más de $70?
premium = np.sum(charges > 70)
print(f"Clientes premium (>$70): {premium} ({premium/len(charges):.1%})")

# ¿Cuál sería el ingreso total si subimos 5% los cargos?
ingreso_actual = np.sum(charges)
ingreso_nuevo = np.sum(charges * 1.05)
print(f"Ingreso actual/mes:  ${ingreso_actual:,.2f}")
print(f"Ingreso con +5%/mes: ${ingreso_nuevo:,.2f}")
print(f"Diferencia:          ${ingreso_nuevo - ingreso_actual:,.2f}")


In [ ]:
# Broadcasting — operar arrays de distinto tamaño
# Escenario: descuento por segmento de tenure

tenure = df['tenure'].values
charges = df['MonthlyCharges'].values

# Definimos descuentos por antigüedad
# < 12 meses → 0% descuento
# 12-24 meses → 5% descuento  
# > 24 meses → 10% descuento
descuentos = np.where(tenure < 12, 0.0,
             np.where(tenure <= 24, 0.05, 0.10))

charges_con_descuento = charges * (1 - descuentos)

print("=== Impacto del programa de descuentos por fidelidad ===")
print(f"Clientes nuevos (<12m):     {np.sum(tenure < 12):,} clientes — sin descuento")
print(f"Clientes medios (12-24m):   {np.sum((tenure >= 12) & (tenure <= 24)):,} clientes — 5% descuento")
print(f"Clientes fieles (>24m):     {np.sum(tenure > 24):,} clientes — 10% descuento")
print(f"\nIngreso actual/mes:         ${np.sum(charges):,.2f}")
print(f"Ingreso con descuentos/mes: ${np.sum(charges_con_descuento):,.2f}")
print(f"Costo del programa:         ${np.sum(charges) - np.sum(charges_con_descuento):,.2f}")

In [ ]:
# Una máscara booleana es un array de True/False
# que usas para filtrar otro array

charges = df['MonthlyCharges'].values
tenure = df['tenure'].values
churn = df['Churn'].values

# Máscara — clientes en riesgo alto
# Definición: nuevos (<12m) + premium (>$70) + mes a mes
contrato = df['Contract'].values
metodo_pago = df['PaymentMethod'].values

mascara_riesgo_alto = (
    (tenure < 12) &
    (charges > 70) &
    (contrato == 'Month-to-month')
)

print("=== Segmento de riesgo alto ===")
print(f"Total clientes:          {len(charges):,}")
print(f"En riesgo alto:          {np.sum(mascara_riesgo_alto):,}")
print(f"Porcentaje:              {np.mean(mascara_riesgo_alto):.1%}")

# Aplicamos la máscara para ver SUS números
charges_riesgo = charges[mascara_riesgo_alto]
churn_riesgo = churn[mascara_riesgo_alto]

print(f"\nCargo promedio segmento: ${np.mean(charges_riesgo):.2f}")
print(f"Churn real en segmento:  {np.mean(churn_riesgo == 'Yes'):.1%}")
print(f"Ingreso mensual en riesgo: ${np.sum(charges_riesgo):,.2f}")

In [ ]:
# Vamos a construir un score de riesgo simple
# combinando múltiples variables — esto es el embrión del modelo del Mes 3

# Extraemos variables numéricas
tenure = df['tenure'].values
charges = df['MonthlyCharges'].values

# Normalizamos entre 0 y 1 — técnica estándar en ML
# Min-Max scaling vectorizado
def minmax_scale(array):
    return (array - np.min(array)) / (np.max(array) - np.min(array))

tenure_norm = minmax_scale(tenure)
charges_norm = minmax_scale(charges)

# Score de riesgo — invertimos tenure (menos tenure = más riesgo)
# y sumamos charges (más cargo = más riesgo si es nuevo)
PESO_TENURE = 0.6   # tenure pesa más — es el predictor más fuerte
PESO_CHARGES = 0.4  # charges pesa menos

risk_score = (1 - tenure_norm) * PESO_TENURE + charges_norm * PESO_CHARGES

print("=== Score de riesgo de churn (0=bajo, 1=alto) ===")
print(f"Score promedio global:        {np.mean(risk_score):.3f}")
print(f"Score promedio — churn Yes:   {np.mean(risk_score[df['Churn'].values == 'Yes']):.3f}")
print(f"Score promedio — churn No:    {np.mean(risk_score[df['Churn'].values == 'No']):.3f}")

print("\n=== Distribución por nivel de riesgo ===")
alto   = np.sum(risk_score > 0.6)
medio  = np.sum((risk_score >= 0.3) & (risk_score <= 0.6))
bajo   = np.sum(risk_score < 0.3)

print(f"Riesgo alto  (>0.6): {alto:,} clientes ({alto/len(risk_score):.1%})")
print(f"Riesgo medio (0.3-0.6): {medio:,} clientes ({medio/len(risk_score):.1%})")
print(f"Riesgo bajo  (<0.3): {bajo:,} clientes ({bajo/len(risk_score):.1%})")

print("\n=== Ingreso en riesgo por nivel ===")
print(f"Riesgo alto:  ${np.sum(charges[risk_score > 0.6]):,.2f}/mes")
print(f"Riesgo medio: ${np.sum(charges[(risk_score >= 0.3) & (risk_score <= 0.6)]):,.2f}/mes")
print(f"Riesgo bajo:  ${np.sum(charges[risk_score < 0.3]):,.2f}/mes")

In [ ]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
# Filtrar clientes con InternetService = No
cond_internet = df["InternetService"] == "No"

# Filtrar clientes con StreamingTV o StreamingMovies activos
cond_streaming = (df["StreamingTV"] == "Yes") | (df["StreamingMovies"] == "Yes")

# Aplicar ambas condiciones
clientes_filtrados = df[cond_internet & cond_streaming]

# Contar resultados
cantidad_clientes = clientes_filtrados.shape[0]
print("Número de clientes:", cantidad_clientes)

print("Número de internet:", cond_internet)
print("Número de streaming:", cond_streaming)


In [ ]:
# Resumen cruzado de InternetService vs StreamingTV y StreamingMovies
tabla_tv = pd.crosstab(df["InternetService"], df["StreamingTV"])
tabla_movies = pd.crosstab(df["InternetService"], df["StreamingMovies"])

print("Distribución StreamingTV:")
print(tabla_tv)

print("\nDistribución StreamingMovies:")
print(tabla_movies)

# Resumen general por tipo de servicio
resumen_servicios = df.groupby("InternetService")[["StreamingTV", "StreamingMovies", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]].describe()

print(resumen_servicios)

In [ ]:
# import pandas as pd

# df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Lista de servicios a analizar
servicios = ["StreamingTV", "StreamingMovies", "OnlineSecurity", 
             "OnlineBackup", "DeviceProtection", "TechSupport"]

# Construir tabla consolidada
tabla_consolidada = pd.DataFrame()

for servicio in servicios:
    conteo = pd.crosstab(df["InternetService"], df[servicio])
    # Tomamos solo la columna "Yes" si existe
    if "Yes" in conteo.columns:
        tabla_consolidada[servicio] = conteo["Yes"]

# Agregar total de clientes por InternetService
tabla_consolidada["Total clientes"] = df["InternetService"].value_counts()

print(tabla_consolidada)


In [ ]:
import numpy as np
import pandas as pd

# Recargamos por si acaso
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Función para detectar outliers con IQR
def detectar_outliers_iqr(serie, nombre):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = serie[(serie < limite_inferior) | (serie > limite_superior)]
    
    print(f"=== Outliers en {nombre} ===")
    print(f"Q1: ${Q1:.2f} | Q3: ${Q3:.2f} | IQR: ${IQR:.2f}")
    print(f"Límite inferior: ${limite_inferior:.2f}")
    print(f"Límite superior: ${limite_superior:.2f}")
    print(f"Outliers encontrados: {len(outliers)} ({len(outliers)/len(serie):.1%})")
    return outliers

outliers_charges = detectar_outliers_iqr(df['MonthlyCharges'], 'MonthlyCharges')
outliers_tenure = detectar_outliers_iqr(df['tenure'], 'tenure')

In [ ]:
# Un cliente que paga $90/mes es normal en Fiber optic
# El mismo cliente en DSL es una anomalía
# Eso es lo que vamos a detectar ahora

print("=== Análisis de anomalías por segmento de Internet ===\n")

for servicio in df['InternetService'].unique():
    segmento = df[df['InternetService'] == servicio]['MonthlyCharges']
    
    Q1 = segmento.quantile(0.25)
    Q3 = segmento.quantile(0.75)
    IQR = Q3 - Q1
    limite_sup = Q3 + 1.5 * IQR
    limite_inf = max(0, Q1 - 1.5 * IQR)  # nunca negativo en negocio
    
    outliers = df[
        (df['InternetService'] == servicio) &
        ((df['MonthlyCharges'] > limite_sup) | 
         (df['MonthlyCharges'] < limite_inf))
    ]
    
    print(f"Segmento: {servicio}")
    print(f"  Rango normal: ${limite_inf:.2f} — ${limite_sup:.2f}")
    print(f"  Outliers:     {len(outliers)} clientes ({len(outliers)/len(segmento):.1%})")
    if len(outliers) > 0:
        print(f"  Churn en outliers: {(outliers['Churn'] == 'Yes').mean():.1%}")
    print()

In [ ]:
# ¿Quiénes son los 342 outliers sin internet?
sin_internet = df[df['InternetService'] == 'No']

Q1 = sin_internet['MonthlyCharges'].quantile(0.25)
Q3 = sin_internet['MonthlyCharges'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = max(0, Q1 - 1.5 * IQR)
limite_sup = Q3 + 1.5 * IQR

outliers_sin_internet = sin_internet[
    (sin_internet['MonthlyCharges'] > limite_sup) |
    (sin_internet['MonthlyCharges'] < limite_inf)
]

print(f"Rango normal Sin Internet: ${limite_inf:.2f} — ${limite_sup:.2f}")
print(f"\n=== Distribución de cargos de los 342 outliers ===")
print(outliers_sin_internet['MonthlyCharges'].describe())

print(f"\n=== ¿Qué servicios tienen estos 342 clientes? ===")
columnas_servicio = ['PhoneService', 'MultipleLines', 'Churn']
print(outliers_sin_internet[columnas_servicio].value_counts())

print(f"\n=== Sus cargos exactos — los más altos ===")
print(outliers_sin_internet[['customerID', 'MonthlyCharges', 
    'PhoneService', 'MultipleLines', 'Contract', 
    'PaymentMethod', 'Churn']]
    .sort_values('MonthlyCharges', ascending=False)
    .head(20))

In [ ]:
# Perfil anómalo: cliente nuevo + sin contrato + 
# paga mucho + método de pago sospechoso
print("=== Perfiles de comportamiento anómalo ===\n")

# Perfil 1 — Cliente fantasma
# Nuevo + sin compromiso + pago no automático
perfil_1 = df[
    (df['tenure'] <= 3) &
    (df['Contract'] == 'Month-to-month') &
    (df['PaymentMethod'] == 'Electronic check') &
    (df['MonthlyCharges'] > df['MonthlyCharges'].quantile(0.75))
]

print(f"Perfil 1 — Cliente fantasma (nuevo+sin contrato+check+premium):")
print(f"  Clientes:          {len(perfil_1)}")
print(f"  Churn:             {(perfil_1['Churn'] == 'Yes').mean():.1%}")
print(f"  Cargo promedio:    ${perfil_1['MonthlyCharges'].mean():.2f}")
print(f"  Ingreso en riesgo: ${perfil_1['MonthlyCharges'].sum():,.2f}/mes")

# Perfil 2 — Cliente invisible
# Lleva tiempo pero nunca acumuló TotalCharges significativo
perfil_2 = df[
    (df['tenure'] > 24) &
    (df['TotalCharges'] < df['TotalCharges'].quantile(0.25)) &
    (df['MonthlyCharges'] > 50)
]

print(f"\nPerfil 2 — Cliente invisible (antiguo+bajo total+cargo medio-alto):")
print(f"  Clientes:          {len(perfil_2)}")
print(f"  Churn:             {(perfil_2['Churn'] == 'Yes').mean():.1%}")
print(f"  Tenure promedio:   {perfil_2['tenure'].mean():.1f} meses")
print(f"  TotalCharges prom: ${perfil_2['TotalCharges'].mean():,.2f}")

# Perfil 3 — El que no encaja en ningún segmento
# Fiber optic pero paga como DSL
perfil_3 = df[
    (df['InternetService'] == 'Fiber optic') &
    (df['MonthlyCharges'] < df[df['InternetService'] == 'DSL']['MonthlyCharges'].quantile(0.75))
]

print(f"\nPerfil 3 — Fiber optic que paga como DSL:")
print(f"  Clientes:          {len(perfil_3)}")
print(f"  Churn:             {(perfil_3['Churn'] == 'Yes').mean():.1%}")
print(f"  Cargo promedio:    ${perfil_3['MonthlyCharges'].mean():.2f}")

In [ ]:
print("=== Perfil 3 — Fiber optic que paga como DSL ===\n")

# Referencia DSL
dsl_q75 = df[df['InternetService'] == 'DSL']['MonthlyCharges'].quantile(0.75)
print(f"Referencia DSL Q75: ${dsl_q75:.2f}")

perfil_3 = df[
    (df['InternetService'] == 'Fiber optic') &
    (df['MonthlyCharges'] < dsl_q75)
]

print(f"\n=== Sus servicios activos ===")
servicios = ['StreamingTV', 'StreamingMovies', 'OnlineSecurity', 
             'OnlineBackup', 'DeviceProtection', 'TechSupport']
for s in servicios:
    activos = (perfil_3[s] == 'Yes').sum()
    print(f"{s}: {activos} de {len(perfil_3)} ({activos/len(perfil_3):.0%})")

print(f"\n=== Contrato y método de pago ===")
print(perfil_3['Contract'].value_counts())
print()
print(perfil_3['PaymentMethod'].value_counts())

print(f"\n=== Tenure ===")
print(perfil_3['tenure'].describe())

In [ ]:
def evaluar_riesgo_cliente(cliente: pd.Series) -> dict:
    """
    Batería de reglas de negocio basadas en EDA.
    Retorna perfil de riesgo con alertas explicables.
    """
    alertas = []
    nivel_riesgo = 0

    # Regla 1 — Cliente fantasma
    if (cliente['tenure'] <= 3 and
        cliente['Contract'] == 'Month-to-month' and
        cliente['PaymentMethod'] == 'Electronic check' and
        cliente['MonthlyCharges'] > 89.85):  # Q3 del dataset
        alertas.append("ALTA: Cliente fantasma — nuevo+sin contrato+check+premium")
        nivel_riesgo += 3

    # Regla 2 — Bandwidth squatter
    servicios = ['StreamingTV', 'StreamingMovies', 'OnlineSecurity',
                 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    servicios_activos = sum(cliente[s] == 'Yes' for s in servicios)

    if (cliente['InternetService'] == 'Fiber optic' and
        cliente['MonthlyCharges'] < 69.90 and
        servicios_activos == 0 and
        cliente['tenure'] < 6):
        alertas.append("ALTA: Bandwidth squatter — Fiber optic sin servicios+precio DSL")
        nivel_riesgo += 3

    # Regla 3 — Método de pago sospechoso + nuevo
    if (cliente['PaymentMethod'] == 'Electronic check' and
        cliente['tenure'] < 12):
        alertas.append("MEDIA: Nuevo cliente con electronic check")
        nivel_riesgo += 1

    # Regla 4 — Contrato débil + premium
    if (cliente['Contract'] == 'Month-to-month' and
        cliente['MonthlyCharges'] > 89.85):
        alertas.append("MEDIA: Cliente premium sin compromiso contractual")
        nivel_riesgo += 1

    # Regla 5 — Combinación letal (lo que validamos el viernes)
    if (cliente['tenure'] < 12 and
        cliente['Contract'] == 'Month-to-month' and
        cliente['InternetService'] == 'Fiber optic'):
        alertas.append("ALTA: Combinación letal — nuevo+mes a mes+Fiber optic")
        nivel_riesgo += 2

    # Clasificación final
    if nivel_riesgo >= 3:
        clasificacion = "🔴 RIESGO ALTO"
    elif nivel_riesgo >= 1:
        clasificacion = "🟡 RIESGO MEDIO"
    else:
        clasificacion = "🟢 RIESGO BAJO"

    return {
        'clasificacion': clasificacion,
        'nivel_riesgo': nivel_riesgo,
        'alertas': alertas,
        'tenure': cliente['tenure'],
        'cargo_mensual': cliente['MonthlyCharges'],
        'churn_real': cliente['Churn']
    }

# Aplicamos a todo el dataset
print("=== Motor de reglas aplicado al dataset completo ===\n")
resultados = df.apply(evaluar_riesgo_cliente, axis=1)

# Conteo por clasificación
alto  = sum(1 for r in resultados if '🔴' in r['clasificacion'])
medio = sum(1 for r in resultados if '🟡' in r['clasificacion'])
bajo  = sum(1 for r in resultados if '🟢' in r['clasificacion'])

print(f"🔴 Riesgo Alto:  {alto:,} clientes ({alto/len(df):.1%})")
print(f"🟡 Riesgo Medio: {medio:,} clientes ({medio/len(df):.1%})")
print(f"🟢 Riesgo Bajo:  {bajo:,} clientes ({bajo/len(df):.1%})")

# Validación — ¿las alertas capturan churn real?
print("\n=== Validación — Churn real por nivel de riesgo ===")
for nivel, emoji in [('ALTO', '🔴'), ('MEDIO', '🟡'), ('BAJO', '🟢')]:
    grupo = [r for r in resultados if emoji in r['clasificacion']]
    if grupo:
        churn_real = sum(1 for r in grupo if r['churn_real'] == 'Yes')
        print(f"{emoji} Riesgo {nivel}: {churn_real/len(grupo):.1%} churn real")

# Ejemplo — ver alertas de los primeros 3 clientes de alto riesgo
print("\n=== Ejemplo — Primeras 3 alertas de riesgo alto ===")
count = 0
for i, r in enumerate(resultados):
    if '🔴' in r['clasificacion'] and count < 3:
        print(f"\nCliente índice {i}:")
        print(f"  Tenure: {r['tenure']} meses | Cargo: ${r['cargo_mensual']:.2f}")
        print(f"  Clasificación: {r['clasificacion']}")
        for alerta in r['alertas']:
            print(f"  ⚠️  {alerta}")
        print(f"  Churn real: {r['churn_real']}")
        count += 1

In [ ]:
# Con historial de clientes — lo que necesitarías en producción
def regla_churner_recurrente(historial_cliente: list) -> dict:
    """
    historial_cliente = lista de contratos del mismo cliente
    identificado por documento/teléfono/email
    """
    if len(historial_cliente) < 2:
        return {"alerta": False}
    
    # Señales de abuso
    reingresos = len(historial_cliente) - 1
    tenure_promedio = sum(c['tenure'] for c in historial_cliente) / len(historial_cliente)
    siempre_promocion = all(c['MonthlyCharges'] < DSL_Q75 
                           for c in historial_cliente)
    
    if reingresos >= 2 and tenure_promedio < 6 and siempre_promocion:
        return {
            "alerta": True,
            "nivel": "🔴 RIESGO ALTO",
            "razon": f"Churner recurrente — {reingresos} reingresos, "
                     f"tenure promedio {tenure_promedio:.1f} meses, "
                     f"siempre en promoción"
        }

In [ ]:
import time

def consultar_banco_central():
    print("Consultando tipo de cambio...")
    time.sleep(2)  # simula una llamada de red de 2 segundos
    print("Respuesta recibida")
    return 1.0

def consultar_score_crediticio():
    print("Consultando score...")
    time.sleep(2)
    print("Score recibido")
    return 750

inicio = time.time()
consultar_banco_central()
consultar_score_crediticio()
print(f"Tiempo total: {time.time() - inicio:.1f}s")

In [ ]:
import asyncio
import time

async def consultar_banco_central():
    print("Consultando tipo de cambio...")
    await asyncio.sleep(2)  # versión async de sleep — no bloquea
    print("Respuesta recibida")
    return 1.0

async def consultar_score_crediticio():
    print("Consultando score...")
    await asyncio.sleep(2)
    print("Score recibido")
    return 750

async def main():
    inicio = time.time()
    
    # Ejecutamos AMBAS al mismo tiempo — no una tras otra
    resultados = await asyncio.gather(
        consultar_banco_central(),
        consultar_score_crediticio()
    )
    
    print(f"Tiempo total: {time.time() - inicio:.1f}s")
    print(f"Resultados: {resultados}")

await main()  # en Jupyter usamos await directo, sin asyncio.run()

In [ ]:
import asyncio
import time

async def consultar_banco_central():
    print("Consultando tipo de cambio...")
    await asyncio.sleep(2)
    print("Respuesta recibida")
    return 1.0

async def consultar_score_crediticio_falla():
    print("Consultando score...")
    await asyncio.sleep(1)  # falla más rápido que el banco central
    raise ConnectionError("Buró de crédito no responde")

async def main_sin_proteccion():
    inicio = time.time()
    try:
        resultados = await asyncio.gather(
            consultar_banco_central(),
            consultar_score_crediticio_falla()
        )
        print(f"Resultados: {resultados}")
    except Exception as e:
        print(f"❌ Error capturado: {e}")
    
    print(f"Tiempo total: {time.time() - inicio:.1f}s")

await main_sin_proteccion()

In [ ]:
import asyncio
import time

async def consultar_banco_central_lento():
    print("🏦 Banco central: iniciando consulta...")
    try:
        await asyncio.sleep(3)  # simula 3 segundos de respuesta
        print("🏦 Banco central: respuesta recibida")
        return 1.0
    except asyncio.CancelledError:
        print("🏦 Banco central: ❌ CANCELADO antes de terminar")
        raise  # importante: siempre re-lanzar la cancelación

async def consultar_score_falla_rapido():
    print("📊 Buró de crédito: iniciando consulta...")
    await asyncio.sleep(1)  # falla mucho más rápido que el banco central
    print("📊 Buró de crédito: ❌ servicio no disponible")
    raise ConnectionError("Buró de crédito no responde")

async def main_taskgroup():
    inicio = time.time()
    try:
        async with asyncio.TaskGroup() as tg:
            tarea_banco = tg.create_task(consultar_banco_central_lento())
            tarea_score = tg.create_task(consultar_score_falla_rapido())
        
        print(f"✅ Ambas completaron: {tarea_banco.result()}, {tarea_score.result()}")
        
    except* ConnectionError as eg:
        print(f"\n❌ Error de conexión capturado: {eg.exceptions}")
    
    print(f"\nTiempo total: {time.time() - inicio:.1f}s")

await main_taskgroup()

In [ ]:
import asyncio
import time

def log(mensaje: str, inicio: float) -> None:
    """Imprime con el tiempo transcurrido desde el inicio — en milisegundos."""
    transcurrido = (time.time() - inicio) * 1000
    print(f"[{transcurrido:6.0f}ms] {mensaje}")


async def consultar_banco_central_lento(inicio: float):
    log("🏦 Banco central: iniciando consulta...", inicio)
    try:
        await asyncio.sleep(3)
        log("🏦 Banco central: respuesta recibida", inicio)
        return 1.0
    except asyncio.CancelledError:
        log("🏦 Banco central: ❌ CANCELADO antes de terminar", inicio)
        raise


async def consultar_score_falla_rapido(inicio: float):
    log("📊 Buró de crédito: iniciando consulta...", inicio)
    await asyncio.sleep(1)
    log("📊 Buró de crédito: ❌ servicio no disponible", inicio)
    raise ConnectionError("Buró de crédito no responde")


async def main_taskgroup_con_timestamps():
    inicio = time.time()
    try:
        async with asyncio.TaskGroup() as tg:
            tarea_banco = tg.create_task(consultar_banco_central_lento(inicio))
            tarea_score = tg.create_task(consultar_score_falla_rapido(inicio))

        log(f"✅ Ambas completaron: {tarea_banco.result()}, {tarea_score.result()}", inicio)

    except* ConnectionError as eg:
        log(f"❌ Error de conexión capturado: {eg.exceptions}", inicio)

    log(f"--- FIN — tiempo total ---", inicio)


await main_taskgroup_con_timestamps()